# SIFLOW · run_2 · Train MDLM SIFLOW head

Trains the velocity head (frozen MDLM backbone) for 20k steps (~3–4h on A100). Checkpoints every 1k steps; the 11h guard stops + checkpoints if needed.

**Runtime:** designed to finish well under one Colab session. Training stops and checkpoints automatically at an 11h guard — if that happens, just re-run this notebook (re-import its checkpoint) and it resumes.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

### Import the previous part(s)

This part needs the output zip(s) you downloaded earlier. Run the cell below — a file picker opens; select **all** of these at once:

- `run_1_data.zip` — tokenized OpenWebText from run_1

*(If a long run here stopped early at the 11h guard, also re-upload **this** part's own checkpoint zip to resume.)* Using Drive instead? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip files listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
!python scripts/train.py --config siflow/config/mdlm.yaml --set \
    data.tokens_path={BASE}/data/owt_gpt2_256.npy \
    out_dir={BASE}/ckpt/mdlm run_id=siflow_mdlm train.total_steps=20000

In [ ]:
from siflow.analysis.curves import load_jsonl, series
import matplotlib.pyplot as plt
rows = load_jsonl(f"{BASE}/ckpt/mdlm/train_log.jsonl")
for k in ("satd", "vel", "mdm"):
    xs, ys = series(rows, k)
    if xs: plt.plot(xs, ys, label=k)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_2_mdlm_ckpt.zip', include=['ckpt/mdlm'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)